# graph-stats

**What does the world airline network look like as a graph?** Not how fast a
search over it runs — Chapters 6 and 7 of the report answer that — but what the
thing being searched actually is: how big, how dense, how lopsided, and how much
of it you can get to from anywhere else.

This is the experiment behind the dataset chapter's statistics. Every number in
`docs/report/02-dataset.md` comes from the `results.json` this notebook writes.

It produces four groups of numbers and two figures:

1. **Size and density** — airports, routes, and how close the network is to one
   where every airport flies to every other.
2. **Degree** — how many routes an airport has, summarised and binned. This is
   where the network turns out to be nothing like uniform.
3. **Reachability** — airports with no departures, no arrivals, or neither.
4. **Components** — how much of the network is mutually reachable.

The data is pinned by `experiment.toml` and verified on load: if a byte of the
snapshot changes, this notebook raises instead of quietly producing a different
answer.

## Setup

Only the first cell differs between Colab and a local checkout.

In [1]:
# In Colab, clone the repository and install the package first:
#   !git clone https://github.com/dgwartney/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima matplotlib
# Then open this notebook from the clone.
try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit('install the package first -- see the comment above') from error

In [2]:
import statistics
from collections import Counter
from pathlib import Path

from flight_planner.experiments import Experiment
from flight_planner.flights import Airport, FlightPlanner, Route
from flight_planner.pathfinding import BFS, SearchObserver

import plots

# The notebook lives in the experiment directory, so the experiment is
# right here. Nothing resolves against a repository root.
experiment = Experiment.open(Path.cwd())
parameters = experiment.parameters

TOP_HUBS = parameters['top_hubs']
DEGREE_BINS = parameters['degree_bins']
CORE_SEED = parameters['core_seed']
TOP_HUBS, DEGREE_BINS, CORE_SEED

(10, [0, 1, 2, 4, 8, 16, 32, 64, 128, 256], 'ATL')

## The data

Opening the snapshot re-hashes every file against the manifest. This experiment
runs on the **whole world network**, unnarrowed — the dataset chapter describes
the graph the rest of the report is measured on, and that is this one.

In [3]:
snapshot = experiment.snapshot
print(snapshot.snapshot_id, snapshot.criteria or '(no criteria: the full network)')

catalog = experiment.catalog()
world = catalog.planner()

AIRPORTS = len(catalog.airports)
ROUTES = len(catalog.routes)
print(f'{AIRPORTS:,} airports / {ROUTES:,} routes')

2026-09-11-bb90a8 (no criteria: the full network)


3,387 airports / 66,332 routes


## 1. How big, and how dense

**Routes are directed and may repeat.** Two airlines flying the same city pair
are two routes, and a route from A to B says nothing about one from B to A. So
there are two different counts of "how connected" this network is, and the gap
between them matters: it is how much parallel service the data carries.

**Density** is edges as a fraction of the most a simple directed graph of this
size could hold, `V x (V - 1)`. It is the number that says whether an
adjacency *list* was the right representation — at this density a matrix would
be almost entirely zeroes, and would cost roughly 11.5 million cells to say so.

In [4]:
pairs = {(route.origin.iata_code, route.destination.iata_code)
         for route in catalog.routes}

possible = AIRPORTS * (AIRPORTS - 1)
size = {
    'airports': AIRPORTS,
    'routes': ROUTES,
    'airport_pairs': len(pairs),
    'parallel_routes': ROUTES - len(pairs),
    'possible_pairs': possible,
    'density': len(pairs) / possible,
    'mean_out_degree': ROUTES / AIRPORTS,
}

print(f"{size['airport_pairs']:,} distinct airport pairs carry {size['routes']:,} routes")
print(f"{size['parallel_routes']:,} routes run parallel to another on the same pair")
print(f"density {size['density']:.4%} of the {size['possible_pairs']:,} pairs a "
      f"complete graph would have")
print(f"mean {size['mean_out_degree']:.1f} routes per airport")

36,717 distinct airport pairs carry 66,332 routes
29,615 routes run parallel to another on the same pair
density 0.3202% of the 11,468,382 pairs a complete graph would have
mean 19.6 routes per airport


## 2. Degree: how many routes an airport has

Counted in a single pass over the routes rather than by asking each airport for
its own. `catalog.routes_from()` scans every route on each call, so airport-by-
airport counting would scan the route table 3,387 times to learn what one pass
already knows.

Degree is directed, so every airport has two: **out** (departures) and **in**
(arrivals).

In [5]:
out_degree = Counter()
in_degree = Counter()
for route in catalog.routes:
    out_degree[route.origin.iata_code] += 1
    in_degree[route.destination.iata_code] += 1

# Indexed from the airport table, not from the counters, so airports with no
# routes at all are counted as zeroes rather than silently missing.
CODES = [airport.iata_code for airport in catalog.airports]
out_series = [out_degree[code] for code in CODES]
in_series = [in_degree[code] for code in CODES]


def summarise(series, counter):
    """Five-number summary of a degree series, naming the airport at the top.

    Args:
        series: One degree per airport, in airport-table order.
        counter: The same data keyed by IATA code, for naming the maximum.

    Returns:
        Mapping of min, median, mean, max and the IATA code holding the max.
    """
    busiest = max(counter, key=counter.get)
    return {
        'min': min(series),
        'median': statistics.median(series),
        'mean': sum(series) / len(series),
        'max': max(series),
        'busiest': busiest,
    }


degree = {'out': summarise(out_series, out_degree),
          'in': summarise(in_series, in_degree)}

for direction, stats in degree.items():
    print(f"{direction:>4}-degree  min {stats['min']:>3}  median {stats['median']:>5}  "
          f"mean {stats['mean']:>5.1f}  max {stats['max']:>4} ({stats['busiest']})")

 out-degree  min   0  median     4  mean  19.6  max  915 (ATL)
  in-degree  min   0  median     4  mean  19.6  max  911 (ATL)


### The shape of it

A mean of 19.6 and a median of 4 are not describing the same network. The mean
is being dragged by a small number of very large airports, which is the single
most important fact about this graph's shape — and the reason the bins below are
powers of two rather than evenly spaced. Even bins would put almost every
airport in the first one.

In [6]:
def bin_degrees(series, bounds):
    """Count airports per degree bucket, with an open-ended final bucket.

    Args:
        series: One degree per airport.
        bounds: Ascending upper bounds, each read as "at most this many".

    Returns:
        List of `(label, count)` in bucket order.
    """
    buckets = []
    previous = -1
    for bound in bounds:
        label = f'{previous + 1}' if bound == previous + 1 else f'{previous + 1}-{bound}'
        buckets.append([label, sum(1 for d in series if previous < d <= bound)])
        previous = bound
    buckets.append([f'{previous + 1}+', sum(1 for d in series if d > previous)])
    return [(label, count) for label, count in buckets]


distribution = {
    'out': bin_degrees(out_series, DEGREE_BINS),
    'in': bin_degrees(in_series, DEGREE_BINS),
}

print(f"{'routes':>10}  {'airports':>8}  {'share':>7}")
for label, count in distribution['out']:
    print(f'{label:>10}  {count:>8,}  {count / AIRPORTS:>6.1%}')

    routes  airports    share
         0        16    0.5%
         1       703   20.8%
         2       582   17.2%
       3-4       595   17.6%
       5-8       495   14.6%
      9-16       330    9.7%
     17-32       231    6.8%
     33-64       182    5.4%
    65-128       129    3.8%
   129-256        83    2.5%
      257+        41    1.2%


### The busiest airports

The out column here is directly checkable against the ad-hoc ranking in
[experiments.md §3](../../docs/experiments.md) — if these disagree, the counting
above is wrong.

In [7]:
by_name = {airport.iata_code: airport for airport in catalog.airports}
hubs = [
    {
        'iata_code': code,
        'name': by_name[code].name,
        'out_degree': out_degree[code],
        'in_degree': in_degree[code],
    }
    for code, _ in out_degree.most_common(TOP_HUBS)
]

for hub in hubs:
    print(f"{hub['iata_code']}  out {hub['out_degree']:>4}  in {hub['in_degree']:>4}  "
          f"{hub['name']}")

ATL  out  915  in  911  Hartsfield Jackson Atlanta International Airport
ORD  out  556  in  548  Chicago O'Hare International Airport
PEK  out  531  in  530  Beijing Capital International Airport
CDG  out  521  in  514  Charles de Gaulle International Airport
LHR  out  520  in  517  London Heathrow Airport
LAX  out  492  in  498  Los Angeles International Airport
FRA  out  489  in  485  Frankfurt Main Airport
DFW  out  469  in  467  Dallas Fort Worth International Airport
JFK  out  454  in  453  John F. Kennedy International Airport
AMS  out  450  in  447  Amsterdam Airport Schiphol


In [8]:
# Dark for the deck, light for the report, from one set of numbers.
out_counts = Counter(out_series)
written = [
    plots.degree_distribution(out_counts, '../../slides/images/degree-distribution.png',
                              mode='dark'),
    plots.degree_distribution(out_counts, '../../docs/images/degree-distribution-light.png',
                              mode='light'),
    plots.top_hubs(hubs, '../../slides/images/top-hubs.png', mode='dark'),
    plots.top_hubs(hubs, '../../docs/images/top-hubs-light.png', mode='light'),
]
for destination in written:
    print(f'wrote {destination}')

wrote ../../slides/images/degree-distribution.png
wrote ../../docs/images/degree-distribution-light.png
wrote ../../slides/images/top-hubs.png
wrote ../../docs/images/top-hubs-light.png


## 3. Reachability

Three ways an airport can be a dead end, and they are not the same thing:

- **no departures** — routes arrive, none leave. A search can reach it and then
  is stuck.
- **no arrivals** — routes leave, none come in. A search can only start there.
- **isolated** — neither. The airport is in the table but in no route.

Counted from the degree counters; no search needed.

In [9]:
no_departures = [code for code in CODES if out_degree[code] == 0]
no_arrivals = [code for code in CODES if in_degree[code] == 0]
isolated = [code for code in no_departures if in_degree[code] == 0]

reachability = {
    'no_departures': len(no_departures),
    'no_arrivals': len(no_arrivals),
    'isolated': len(isolated),
}
print(reachability)

{'no_departures': 16, 'no_arrivals': 7, 'isolated': 0}


## 4. Components: how much of the network hangs together

This is the one question here that needs a traversal — and the package already
has one, so this notebook does not write another.

The trick is that **a search with an unreachable goal visits everything it can
reach**. `BFS` only ever compares the goal for equality; it never asks the graph
for the goal's edges. Hand it a sentinel airport that is not in the graph and it
drains its queue over the whole reachable set, then reports an infinite cost —
exactly the documented behaviour for an unreachable destination.

That turns `BFS` into a reachability probe, and an observer collects the answer.
`SearchObserver` exists for questions nobody asked when it was written; this is
one. BFS marks a vertex visited when it is *queued*, so overriding the single
`on_push` hook captures every reachable airport.

In [10]:
SENTINEL = Airport('~~~')  # not in the graph, and no real IATA code looks like this


class ReachableSet(SearchObserver):
    """Collects every vertex a search reaches.

    BFS marks a vertex visited at push time, so the pushes are the reachable
    set. The start is never pushed by the loop, so it is seeded here.

    Attributes:
        seen: Every airport reached, including the start.
    """

    def __init__(self, start):
        """Begin a collection rooted at `start`.

        Args:
            start: The airport the search departs from.
        """
        self.seen = {start}

    def on_push(self, vertex, priority):
        """Record a queued airport.

        Args:
            vertex: The airport being queued.
            priority: Its hop depth, unused here.
        """
        self.seen.add(vertex)


def reachable_from(planner, start):
    """Return every airport reachable from `start`, following edge direction.

    Args:
        planner: The graph to search.
        start: The airport to depart from.

    Returns:
        Set of reachable `Airport`, including `start` itself.

    Raises:
        AssertionError: If the search terminated early, which would mean the
            sentinel was somehow reached and the result is not a full closure.
    """
    observer = ReachableSet(start)
    result = planner.search(start, SENTINEL, BFS(), observer=observer)
    assert result.cost == float('inf'), 'the sentinel must never be reachable'
    return observer.seen


seed = world.find_airport(CORE_SEED)
forward = reachable_from(world, seed)
print(f'{len(forward):,} of {AIRPORTS:,} airports are reachable from {CORE_SEED}')

3,341 of 3,387 airports are reachable from ATL


### Which airports can reach the seed

The mirror question needs the same searches run against reversed routes. Both
graphs below are built from the catalog's own routes through the ordinary
`Route` constructor — the only new thing is which way round the endpoints go.

In [11]:
def rebuilt(edges):
    """Build a planner over the same airports with a different edge set.

    Args:
        edges: The routes to add.

    Returns:
        A `FlightPlanner` holding every catalog airport and the given routes.
    """
    planner = FlightPlanner()
    for airport in catalog.airports:
        planner.add_vertex(airport)
    for route in edges:
        planner.add_edge(route)
    return planner


def flipped(route):
    """Return the same leg flown the other way.

    Args:
        route: The route to reverse.

    Returns:
        A `Route` with origin and destination exchanged.
    """
    return Route(route.destination, route.origin, route.weight,
                 route.airline, route.flight_number)


transpose = rebuilt([flipped(route) for route in catalog.routes])
undirected = rebuilt(list(catalog.routes) + [flipped(route) for route in catalog.routes])

backward = reachable_from(transpose, seed)
core = forward & backward
print(f'{len(backward):,} airports can reach {CORE_SEED}')
print(f'{len(core):,} airports are mutually reachable with it')

3,336 airports can reach ATL
3,318 airports are mutually reachable with it


### Weakly connected components

Ignoring direction entirely: which airports are joined to which at all. One BFS
per component, seeded from the first airport not yet accounted for.

In [12]:
seen = set()
component_sizes = []
for airport in catalog.airports:
    if airport in seen:
        continue
    members = reachable_from(undirected, airport)
    component_sizes.append(len(members))
    seen |= members

component_sizes.sort(reverse=True)
components = {
    'weak_count': len(component_sizes),
    'weak_sizes': component_sizes,
    'largest_weak': component_sizes[0],
    'strong_core': len(core),
    'strong_core_seed': CORE_SEED,
    'reachable_from_seed': len(forward),
    'can_reach_seed': len(backward),
}

print(f"{components['weak_count']} weakly connected components, "
      f"sizes {component_sizes[:6]}{' ...' if len(component_sizes) > 6 else ''}")
print(f"largest holds {components['largest_weak'] / AIRPORTS:.1%} of all airports")

8 weakly connected components, sizes [3359, 10, 4, 4, 4, 2] ...
largest holds 99.2% of all airports


## Record

`experiment.record` writes `results.json` next to this notebook, carrying the
snapshot's identity, criteria and source commit alongside the numbers — so a
result can always be traced to the data that produced it.

In [13]:
path = experiment.record({
    'size': size,
    'degree': degree,
    'degree_distribution': distribution,
    'top_hubs': hubs,
    'reachability': reachability,
    'components': components,
}, catalog=catalog)
print(f'recorded -> {path}')

recorded -> /Users/dgwartney/git/traiectoria-optima/.claude/worktrees/EDA/experiments/graph-stats/results.json


## What this shows

**The network is sparse and enormously lopsided.** Under a tenth of a percent of
the airport pairs a complete graph would have are actually flown, which is what
makes an adjacency list the right structure and a matrix the wrong one. Within
that sparse graph, degree spans three orders of magnitude: the median airport has
a handful of routes and the busiest has hundreds. The mean is more than four
times the median, and every claim about "a typical airport" should be read with
that in mind.

**Almost all of it hangs together.** The overwhelming majority of airports sit in
one weakly connected component, and nearly as many sit in a single mutually
reachable core. That is what makes the long-haul queries in Chapter 7 meaningful:
a route between two randomly chosen busy airports almost always exists, so the
searches being compared are doing real work rather than discovering "no route".

**The handful that do not connect are worth knowing about.** Airports with
arrivals and no departures, or no routes at all, are the cases that make
`find_shortest_route` return infinite cost — the behaviour Chapter 5's
disconnected-graph test pins down.